## A complete 15-state ESKF step (propagate + camera update), with code

Below we go through **one full 15-state Error-State Kalman Filter step** — IMU
propagate followed by a monocular-VO update — with **numbers and code side by
side**. Each block of math is immediately followed by a tiny code cell that
performs *exactly that step*, so you can read the equation and the
implementation as a pair.

We assume monocular VO gives a relative pose $(R_{vo}, t_{vo})$ between two
camera frames, and that the translation has been rescaled to $|t_{vo}|=1$
(pseudo-metric — scale is not actually observable from a single camera). For
simplicity we take the world pose at time $k$ to be the identity, so the
composed "absolute" measurement at $k{+}1$ is just $(R_{vo}, t_{vo})$ itself.

---

# 1) State and ordering

### Nominal (stored)
$$
\hat x = (\hat p,\ \hat v,\ \hat R,\ \hat b_g,\ \hat b_a)
$$

### Error state (for the covariance only)
$$
\delta x = (\delta p,\ \delta v,\ \delta\theta,\ \delta b_g,\ \delta b_a)\in\mathbb{R}^{15}
$$

Ordering used everywhere below:
$$
[\delta p(0{:}3),\ \delta v(3{:}6),\ \delta\theta(6{:}9),\ \delta b_g(9{:}12),\ \delta b_a(12{:}15)]
$$


### Setup: `skew`, `Exp`, `Log` on $SO(3)$

We will need three Lie-group helpers throughout:

- $[\cdot]_\times$: the $3\times 3$ skew-symmetric matrix of a 3-vector,
- $\operatorname{Exp}: \mathbb{R}^3 \to SO(3)$ (Rodrigues' formula),
- $\operatorname{Log}: SO(3) \to \mathbb{R}^3$ (inverse map).


In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

# 3x3 skew-symmetric matrix from a 3-vector
def skew(v):
    return np.array([[   0, -v[2],  v[1]],
                     [v[2],     0, -v[0]],
                     [-v[1],  v[0],    0]])

# Lie-group exponential: R^3 -> SO(3) via Rodrigues
def Exp_SO3(phi):
    theta = np.linalg.norm(phi)
    if theta < 1e-12:
        return np.eye(3) + skew(phi)
    K = skew(phi / theta)
    return np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)

# Lie-group logarithm: SO(3) -> R^3
def Log_SO3(R):
    cos_theta = np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    w = np.array([R[2, 1] - R[1, 2],
                  R[0, 2] - R[2, 0],
                  R[1, 0] - R[0, 1]])
    if abs(theta) < 1e-12:
        return 0.5 * w
    return (theta / (2 * np.sin(theta))) * w


# 2) Given values (time $k \to k{+}1$)

$$
\Delta t = 0.1\ \text{s},\quad g=\begin{bmatrix}0\\0\\-9.81\end{bmatrix}
$$

Nominal at $k$:
$$
\hat p=\begin{bmatrix}0\\0\\0\end{bmatrix},\quad
\hat v=\begin{bmatrix}1\\0\\0\end{bmatrix},\quad
\hat R=I
$$

$$
\hat b_g=\begin{bmatrix}0.02\\-0.01\\0.005\end{bmatrix}\ \text{rad/s},\quad
\hat b_a=\begin{bmatrix}0.1\\-0.05\\0.02\end{bmatrix}\ \text{m/s}^2
$$

IMU readings:
$$
\omega_m=\begin{bmatrix}0\\0\\1\end{bmatrix}\ \text{rad/s},\quad
a_m=\begin{bmatrix}0.4\\0\\9.81\end{bmatrix}\ \text{m/s}^2
$$

Bias-corrected angular velocity and specific force:
$$
\omega=\omega_m-\hat b_g,\qquad f=a_m-\hat b_a
$$


In [2]:
dt = 0.1
g  = np.array([0.0, 0.0, -9.81])

# Nominal state at time k
p   = np.zeros(3)
v   = np.array([1.0, 0.0, 0.0])
R   = np.eye(3)
b_g = np.array([0.02, -0.01, 0.005])
b_a = np.array([0.1,  -0.05, 0.02])

# IMU readings (raw)
omega_m = np.array([0.0, 0.0, 1.0])
a_m     = np.array([0.4, 0.0, 9.81])

# Bias-corrected
omega = omega_m - b_g
f     = a_m     - b_a
print("omega =", omega)
print("f     =", f)


omega = [-0.02   0.01   0.995]
f     = [0.3  0.05 9.79]


# 3) Nominal propagation

Rotation increment over $\Delta t$:
$$
\phi=\omega\Delta t,\qquad
\hat R^+ = \hat R\,\operatorname{Exp}(\phi^\wedge)
$$

Acceleration in world frame, velocity, position:
$$
a_w = \hat R^+ f + g,\quad
\hat v^+ = \hat v + a_w\Delta t,\quad
\hat p^+ = \hat p + \hat v\Delta t + \tfrac12 a_w\Delta t^2
$$

Biases are modelled as random walks, so the **nominal** does not change in
prediction (only their covariance grows):
$$
\hat b_g^+=\hat b_g,\qquad \hat b_a^+=\hat b_a
$$


In [3]:
phi   = omega * dt                    # so(3) increment
R_pl  = R @ Exp_SO3(phi)              # R+
a_w   = R_pl @ f + g                  # world-frame acceleration
v_pl  = v + a_w * dt                  # v+
p_pl  = p + v * dt + 0.5 * a_w * dt**2  # p+

# bias nominal does not change in prediction
b_g_pl = b_g.copy()
b_a_pl = b_a.copy()

print("phi   =", phi)
print("R+    =\n", R_pl)
print("a_w   =", a_w)
print("v+    =", v_pl)
print("p+    =", p_pl)


phi   = [-0.002   0.001   0.0995]
R+    =
 [[ 0.995053 -0.099337  0.000899]
 [ 0.099335  0.995052  0.002046]
 [-0.001098 -0.001947  0.999998]]
a_w   = [ 0.30235   0.099587 -0.020451]
v+    = [ 1.030235  0.009959 -0.002045]
p+    = [ 0.101512  0.000498 -0.000102]


# 4) Monocular VO gives a relative pose

VO produces the relative motion between camera frames $k$ and $k{+}1$:

- relative rotation $R_{vo} = \operatorname{Exp}([0,0,0.09]^\wedge)$ (yaw $0.09$ rad),
- relative translation **direction**, rescaled to $|t_{vo}|=1$: $t_{vo}=[1,0,0]^T$.

We take the world pose at $k$ to be the identity, so the pseudo-absolute
measurement at $k{+}1$ is simply
$$
R_{meas}=R_{vo},\qquad p_{meas}=t_{vo}.
$$


In [4]:
R_vo  = Exp_SO3(np.array([0.0, 0.0, 0.09]))
t_vo  = np.array([1.0, 0.0, 0.0])

R_meas = R_vo
p_meas = t_vo
print("R_meas =\n", R_meas)
print("p_meas =", p_meas)


R_meas =
 [[ 0.995953 -0.089879  0.      ]
 [ 0.089879  0.995953  0.      ]
 [ 0.        0.        1.      ]]
p_meas = [1. 0. 0.]


# 5) Measurement residual (6D: position + orientation)

$$
r_p     = p_{meas} - \hat p^+,\qquad
r_\theta = \operatorname{Log}\!\big(R_{meas}\,(\hat R^+)^T\big)
$$

Stack into one residual:
$$
r=\begin{bmatrix} r_p\\ r_\theta \end{bmatrix}\in\mathbb{R}^6.
$$


In [5]:
r_p     = p_meas - p_pl
r_theta = Log_SO3(R_meas @ R_pl.T)
r       = np.concatenate([r_p, r_theta])
print("r_p     =", r_p)
print("r_theta =", r_theta)


r_p     = [ 0.898488 -0.000498  0.000102]
r_theta = [ 0.002042 -0.000909 -0.0095  ]


# 6) Measurement Jacobian $H$

Position depends linearly on $\delta p$; orientation depends on $\delta\theta$.
All other error-state blocks have zero influence on this measurement:

$$
H=\begin{bmatrix}
I_3 & 0 & 0   & 0 & 0\\
0   & 0 & I_3 & 0 & 0
\end{bmatrix}\in\mathbb{R}^{6\times 15}.
$$


In [6]:
H = np.zeros((6, 15))
H[0:3, 0:3] = np.eye(3)   # r_p   <- delta p
H[3:6, 6:9] = np.eye(3)   # r_th  <- delta theta
print("H =\n", H)


H =
 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]]


# 7) Error-state transition Jacobian $F$

Discrete-time (Euler) linearisation of the error dynamics:

$$
\begin{aligned}
\delta p^+      &= \delta p + \delta v\,\Delta t \\
\delta v^+      &= \delta v - \hat R^+[f]_\times\,\delta\theta\,\Delta t - \hat R^+\,\delta b_a\,\Delta t \\
\delta\theta^+  &= (I-[\omega]_\times\Delta t)\,\delta\theta - \delta b_g\,\Delta t \\
\delta b_g^+    &= \delta b_g \\
\delta b_a^+    &= \delta b_a
\end{aligned}
$$

Non-identity blocks of $F$:

- $F_{pv}=I\Delta t$
- $F_{v\theta}=-\hat R^+[f]_\times\Delta t$
- $F_{v b_a}=-\hat R^+\Delta t$
- $F_{\theta\theta}=I-[\omega]_\times\Delta t$
- $F_{\theta b_g}=-I\Delta t$


In [7]:
F = np.eye(15)
F[0:3,  3:6 ] =  np.eye(3) * dt                # F_pv
F[3:6,  6:9 ] = -R_pl @ skew(f) * dt           # F_v_theta
F[3:6, 12:15] = -R_pl * dt                     # F_v_ba
F[6:9,  6:9 ] =  np.eye(3) - skew(omega) * dt  # F_theta_theta
F[6:9,  9:12] = -np.eye(3) * dt                # F_theta_bg
# F_bg_bg and F_ba_ba are identity (random walk), already set by np.eye(15)
print("F (non-trivial blocks):")
print("F[0:3, 3:6]   =\n", F[0:3, 3:6])
print("F[3:6, 6:9]   =\n", F[3:6, 6:9])
print("F[6:9, 6:9]   =\n", F[6:9, 6:9])


F (non-trivial blocks):
F[0:3, 3:6]   =
 [[0.1 0.  0. ]
 [0.  0.1 0. ]
 [0.  0.  0.1]]
F[3:6, 6:9]   =
 [[ 0.097255  0.97413  -0.007955]
 [-0.974146  0.097187  0.029355]
 [ 0.006906 -0.031075 -0.000053]]
F[6:9, 6:9]   =
 [[ 1.      0.0995 -0.001 ]
 [-0.0995  1.     -0.002 ]
 [ 0.001   0.002   1.    ]]


# 8) Covariance prediction: $P^- = F P F^T + Q$

Initial covariance — diagonal blocks plus two **cross-correlations** that
make the biases reachable by the camera update:

- $P_{pp}=0.01\,I,\ P_{vv}=0.04\,I,\ P_{\theta\theta}=0.0025\,I$
- $P_{b_gb_g}=10^{-4}\,I,\ P_{b_ab_a}=10^{-2}\,I$
- $P_{\theta b_g}=5\cdot10^{-5}\,I$  *(orientation $\leftrightarrow$ gyro bias)*
- $P_{v b_a}=2\cdot10^{-3}\,I$  *(velocity $\leftrightarrow$ accel bias)*

Process-noise (simple diagonal discretization):

- gyro white $\sigma_g=0.01$, accel white $\sigma_a=0.1$
- gyro RW $\sigma_{bg}=10^{-3}$, accel RW $\sigma_{ba}=10^{-2}$

$$
Q_{vv}=\sigma_a^2\Delta t\,I,\ \
Q_{\theta\theta}=\sigma_g^2\Delta t\,I,\ \
Q_{b_g b_g}=\sigma_{bg}^2\Delta t\,I,\ \
Q_{b_a b_a}=\sigma_{ba}^2\Delta t\,I.
$$


In [8]:
# --- prior covariance P ---
P = np.zeros((15, 15))
P[0:3,   0:3  ] = 0.01   * np.eye(3)   # P_pp
P[3:6,   3:6  ] = 0.04   * np.eye(3)   # P_vv
P[6:9,   6:9  ] = 0.0025 * np.eye(3)   # P_theta_theta
P[9:12,  9:12 ] = 1e-4   * np.eye(3)   # P_bg_bg
P[12:15, 12:15] = 1e-2   * np.eye(3)   # P_ba_ba

# cross-correlations (and their transposes)
P[6:9,   9:12 ] = 5e-5   * np.eye(3); P[9:12,  6:9 ] = P[6:9, 9:12].T
P[3:6,  12:15 ] = 2e-3   * np.eye(3); P[12:15, 3:6 ] = P[3:6, 12:15].T

# --- process noise Q ---
sigma_g, sigma_a, sigma_bg, sigma_ba = 0.01, 0.1, 1e-3, 1e-2
Q = np.zeros((15, 15))
Q[3:6,   3:6  ] = sigma_a **2 * dt * np.eye(3)
Q[6:9,   6:9  ] = sigma_g **2 * dt * np.eye(3)
Q[9:12,  9:12 ] = sigma_bg**2 * dt * np.eye(3)
Q[12:15, 12:15] = sigma_ba**2 * dt * np.eye(3)

# --- prediction ---
P_minus = F @ P @ F.T + Q
print("diag(P-) block diagonals:")
print("  P_pp        :", np.diag(P_minus)[0:3])
print("  P_vv        :", np.diag(P_minus)[3:6])
print("  P_theta_th  :", np.diag(P_minus)[6:9])
print("  P_bg_bg     :", np.diag(P_minus)[9:12])
print("  P_ba_ba     :", np.diag(P_minus)[12:15])


diag(P-) block diagonals:
  P_pp        : [0.0104 0.0104 0.0104]
  P_vv        : [0.043098 0.0431   0.040703]
  P_theta_th  : [0.002526 0.002526 0.002501]
  P_bg_bg     : [0.0001 0.0001 0.0001]
  P_ba_ba     : [0.01001 0.01001 0.01001]


# 9) Measurement noise $R_m$

Because we've forced $|t_{vo}|=1$ rather than using a real metric scale, the
position part of the measurement is a **pseudo-measurement**. We absorb that
fact by giving it a large diagonal noise. The rotation part is genuine VO,
so a much smaller noise:

$$
R_m=\mathrm{diag}(0.25,0.25,0.25,\ 0.01,0.01,0.01).
$$


In [9]:
R_m = np.diag([0.25, 0.25, 0.25, 0.01, 0.01, 0.01])
print("R_m =\n", R_m)


R_m =
 [[0.25 0.   0.   0.   0.   0.  ]
 [0.   0.25 0.   0.   0.   0.  ]
 [0.   0.   0.25 0.   0.   0.  ]
 [0.   0.   0.   0.01 0.   0.  ]
 [0.   0.   0.   0.   0.01 0.  ]
 [0.   0.   0.   0.   0.   0.01]]


# 10) Kalman update — and where the bias corrections come from

$$
S = H P^- H^T + R_m,\qquad
K = P^- H^T S^{-1},\qquad
\boxed{\delta\hat x = K\,r}
$$

The bias updates $\delta\hat b_g$ and $\delta\hat b_a$ are non-zero **only
because** of the cross-covariances $P_{\theta b_g}$ and $P_{v b_a}$ that we
seeded in section 8 (and that the propagation $FPF^T$ keeps populating). A
diagonal $P$ would leave the biases frozen.


In [10]:
S = H @ P_minus @ H.T + R_m
K = P_minus @ H.T @ np.linalg.inv(S)

dx = K @ r
delta_p     = dx[0:3]
delta_v     = dx[3:6]
delta_theta = dx[6:9]
delta_bg    = dx[9:12]
delta_ba    = dx[12:15]

print("delta_p     =", delta_p)
print("delta_v     =", delta_v)
print("delta_theta =", delta_theta)
print("delta_bg    =", delta_bg)
print("delta_ba    =", delta_ba)


delta_p     = [ 0.035884 -0.00002   0.000004]
delta_v     = [ 0.013649 -0.000496  0.000009]
delta_theta = [ 0.000411 -0.000183 -0.001901]
delta_bg    = [ 0.000007 -0.000002 -0.00003 ]
delta_ba    = [ 0.00069 -0.       0.     ]


# 11) Inject the correction back into the nominal state

Linear states get added; the rotation is composed on the manifold so it stays
in $SO(3)$:

$$
\hat p \leftarrow \hat p^+ + \delta\hat p,\quad
\hat v \leftarrow \hat v^+ + \delta\hat v,\quad
\hat R \leftarrow \hat R^+\,\operatorname{Exp}((\delta\hat\theta)^\wedge),
$$
$$
\hat b_g \leftarrow \hat b_g + \delta\hat b_g,\quad
\hat b_a \leftarrow \hat b_a + \delta\hat b_a.
$$

Covariance update (Joseph form — numerically stable):
$$
P \leftarrow (I-KH)\,P^-(I-KH)^T + K R_m K^T.
$$

Reset is only conceptual: by construction the **mean** of the new error state
is zero immediately after injection.


In [11]:
p_new   = p_pl + delta_p
v_new   = v_pl + delta_v
R_new   = R_pl @ Exp_SO3(delta_theta)
b_g_new = b_g  + delta_bg
b_a_new = b_a  + delta_ba

I15 = np.eye(15)
P_new = (I15 - K @ H) @ P_minus @ (I15 - K @ H).T + K @ R_m @ K.T

print("p_new   =", p_new)
print("v_new   =", v_new)
print("R_new   =\n", R_new)
print("b_g_new =", b_g_new)
print("b_a_new =", b_a_new)
print("P_new symmetric?", np.allclose(P_new, P_new.T))


p_new   = [ 0.137396  0.000478 -0.000098]
v_new   = [ 1.043884  0.009463 -0.002036]
R_new   =
 [[ 0.995241 -0.097445  0.000757]
 [ 0.097444  0.99524   0.001619]
 [-0.000911 -0.001537  0.999998]]
b_g_new = [ 0.020007 -0.010002  0.00497 ]
b_a_new = [ 0.10069 -0.05     0.02   ]
P_new symmetric? True


---

## Caveat: forcing $|t_{vo}|=1$

With a single camera the translation from VO is only a **direction** — the
metric scale is unobservable without extra information (stereo baseline,
IMU-VIO bootstrap, known landmark, etc.). Rescaling to $|t|=1$ makes it a
**pseudo-measurement**, which is fine as long as you reflect that with a
large position-component of $R_m$ (here $0.25$). With a small $R_m$ the
filter would happily over-trust a fake metric scale and drift hard.

A more honest monocular update is a **bearing constraint**: drop the
translation-magnitude information entirely and only update on
$(R_{vo},\ \hat t_{vo})$ where $\hat t_{vo}$ is a unit direction. That's a
clean follow-up: same skeleton, different residual + Jacobian.
